# Final Pipeline Evaluation

## Objective

This notebook evaluates the end-to-end Murawa pipeline as a demonstration system. The goal is not to repeat the model benchmarks from the comparison notebook, but to assess the complete flow from raw video or image input to annotated output: detection, ByteTrack tracking, jersey-color team assignment, minimap entity extraction, and the rendered output artifacts.

The evaluation is qualitative and practical. We look at what the system does well on real footage, where it fails, and what the current demo version can realistically be expected to deliver.

## 1. Setup

In [ ]:
import json
import math
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from IPython.display import display, Image as IPImage

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "font.size": 11,
})

TEAM_A_COLOR = "#e05555"
TEAM_B_COLOR = "#5599e0"
REFEREE_COLOR = "#55dede"
UNKNOWN_COLOR = "#aaaaaa"

TEAM_COLORS = {
    "team_a": TEAM_A_COLOR,
    "team_b": TEAM_B_COLOR,
    "referee": REFEREE_COLOR,
    "unknown": UNKNOWN_COLOR,
}

PROJECT_ROOT = Path("..")
PREDICTIONS_DIR = PROJECT_ROOT / "outputs" / "predictions"
VIDEOS_DIR = PROJECT_ROOT / "outputs" / "videos"

## 2. Pipeline Description

The Murawa pipeline converts a video clip or a single frame into annotated outputs. The entry points are `analyze_frame` and `analyze_match` in `murawa.services.analysis.pipeline`. Both write a `prediction_summary.json` to `outputs/predictions/<run_name>/<analysis_id>/` and the match mode additionally writes an annotated video to `outputs/videos/`.

The full match pipeline runs in seven steps:

1. **Input validation** - the video is checked against allowed suffixes and a maximum duration of 60 seconds (configurable in `configs/project.yaml`).
2. **Frame extraction** - frames are sampled at the configured rate (default 3 FPS, range 1-24) using OpenCV into a temporary directory.
3. **Model inference** - the selected checkpoint (YOLO or RF-DETR) runs on every sampled frame and returns bounding boxes with class labels and confidence scores for the four classes: `player`, `goalkeeper`, `referee`, `ball`.
4. **Tracking** - ByteTrack (via the `supervision` library) assigns persistent track IDs across frames. Each frame batch is passed to the tracker sequentially.
5. **Per-frame team assignment** - jersey colors are extracted from a cropped torso region of each detected player, grass pixels are masked out in HSV space, and the remaining pixels' median BGR value is computed. Players in a frame are clustered into two groups using a K-means-style procedure initialized at the two most distant colors in LAB space.
6. **Clip-level team stabilization** - jersey color prototypes are built from all sampled frames to prevent the per-frame clustering from swapping `team_a`/`team_b` labels between frames. Detections are then relabeled against the stable prototypes. A 15-second sliding window further smooths class and team labels based on track history.
7. **Rendering** - an annotated video is written (WebM for streaming, MP4 for download) with colored bounding boxes, team labels, a legend, and a detection count overlay. Per-frame minimap entities are computed as a list of positions and roles in JSON, not as a graphical layer on the video.

The diagram below summarizes the data flow.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.2))
ax.set_xlim(0, 12)
ax.set_ylim(0, 2.2)
ax.axis("off")

steps = [
    ("Input\nvideo / frame", 0.5),
    ("Frame\nextraction", 2.0),
    ("Model\ninference", 3.5),
    ("ByteTrack\ntracking", 5.0),
    ("Team\nassignment", 6.5),
    ("Clip\nstabilization", 8.0),
    ("Render\noutputs", 9.5),
    ("prediction_summary\n+ video", 11.2),
]

box_w, box_h = 1.2, 0.7
for label, x in steps:
    is_output = "prediction" in label
    facecolor = "#dbeafe" if not is_output else "#dcfce7"
    edgecolor = "#3b82f6" if not is_output else "#16a34a"
    rect = mpatches.FancyBboxPatch(
        (x - box_w / 2, 1.1 - box_h / 2), box_w, box_h,
        boxstyle="round,pad=0.05", linewidth=1.2,
        edgecolor=edgecolor, facecolor=facecolor
    )
    ax.add_patch(rect)
    ax.text(x, 1.1, label, ha="center", va="center", fontsize=7.5, linespacing=1.4)

for i in range(len(steps) - 1):
    x0 = steps[i][1] + box_w / 2
    x1 = steps[i + 1][1] - box_w / 2
    ax.annotate("", xy=(x1, 1.1), xytext=(x0, 1.1),
                arrowprops=dict(arrowstyle="->", color="#6b7280", lw=1.2))

plt.title("Murawa end-to-end pipeline", fontsize=11, pad=8)
plt.tight_layout()
plt.show()

## 3. Test Data

Evaluation material comes from two sources:

- **Saved prediction summaries** stored in `outputs/predictions/`. Each analysis directory contains a `prediction_summary.json` with the full result payload including per-detection metadata, tracking information, team assignment, and minimap entities.
- **Annotated output videos** stored in `outputs/videos/`. These are the visual output of the match pipeline.

The code below discovers all available summaries and builds a working index.

In [ ]:
def load_prediction_summaries(predictions_dir: Path) -> list[dict]:
    summaries = []
    for summary_path in sorted(predictions_dir.rglob("prediction_summary.json")):
        try:
            payload = json.loads(summary_path.read_text(encoding="utf-8"))
            payload["_path"] = str(summary_path)
            summaries.append(payload)
        except Exception as e:
            print(f"Could not load {summary_path}: {e}")
    return summaries


summaries = load_prediction_summaries(PREDICTIONS_DIR)
print(f"Found {len(summaries)} prediction summary file(s) in outputs/predictions/")

In [ ]:
if summaries:
    rows = []
    for s in summaries:
        stats = s.get("stats", {})
        tracking = s.get("tracking", {})
        rows.append({
            "mode": s.get("mode", "?"),
            "model": s.get("model", "?"),
            "dataset_variant": s.get("dataset_variant", "?"),
            "run_name": s.get("resolved_run_name", "")[-30:],
            "sampled_frames": s.get("sampled_frames", 0),
            "total_detections": stats.get("total_detections", 0),
            "track_count": stats.get("track_count", tracking.get("track_count", 0)),
            "mean_confidence": round(stats.get("mean_confidence", 0.0), 3),
            "sample_fps": s.get("sample_fps", ""),
        })
    df_index = pd.DataFrame(rows)
    display(df_index)
else:
    print("No summaries available. Run the pipeline on at least one video or frame to populate this section.")
    print("Example: python scripts/predict.py --model yolo --dataset-variant base --mode match --input-path <video>")

## 4. Evaluation Criteria

There are no ground-truth annotations for the demo clips, so the evaluation is qualitative rather than metric-based. We assess the four pipeline components using the criteria below.

| Component | What we check |
|---|---|
| Detection | Detection count per class, confidence distribution, presence of obvious false positives or misses |
| Tracking | Number of unique track IDs relative to expected objects, identity switches visible in the data |
| Team assignment | Team size balance, fraction of `unknown` assignments, consistency across frames |
| Minimap | Number of entities populated, class distribution, coverage relative to tracked players |

## 5. Detection Results

In [ ]:
match_summaries = [s for s in summaries if s.get("mode") == "match"]
frame_summaries = [s for s in summaries if s.get("mode") == "frame"]

print(f"Match mode summaries: {len(match_summaries)}")
print(f"Frame mode summaries: {len(frame_summaries)}")

focus = match_summaries[0] if match_summaries else (frame_summaries[0] if frame_summaries else None)
if focus:
    print(f"Using summary: {focus.get('resolved_run_name', '?')} | mode={focus.get('mode')} | {focus.get('_path', '')}")
else:
    print("No summaries loaded. The analysis cells below will be skipped.")

In [ ]:
if focus:
    stats = focus.get("stats", {})
    classes = stats.get("classes", {})

    if classes:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))

        ax = axes[0]
        sorted_classes = sorted(classes.items(), key=lambda x: x[1], reverse=True)
        labels = [c for c, _ in sorted_classes]
        counts = [n for _, n in sorted_classes]
        class_palette = {"player": "#3b82f6", "goalkeeper": "#f59e0b",
                         "referee": "#10b981", "ball": "#ef4444"}
        colors = [class_palette.get(lbl, "#94a3b8") for lbl in labels]
        bars = ax.bar(labels, counts, color=colors, edgecolor="white", linewidth=0.8)
        ax.bar_label(bars, padding=3, fontsize=10)
        ax.set_title("Total detections by class")
        ax.set_ylabel("Detections")
        ax.set_xlabel("")

        ax2 = axes[1]
        detections = focus.get("detections", [])
        confidences_by_class = defaultdict(list)
        for det in detections:
            c = str(det.get("class", "unknown"))
            conf = det.get("confidence")
            if isinstance(conf, (int, float)):
                confidences_by_class[c].append(float(conf))

        all_confs = [conf for confs in confidences_by_class.values() for conf in confs]
        if all_confs:
            ax2.hist(all_confs, bins=20, color="#6366f1", edgecolor="white", linewidth=0.5, alpha=0.85)
            ax2.axvline(np.mean(all_confs), color="#dc2626", linestyle="--", linewidth=1.5,
                        label=f"mean = {np.mean(all_confs):.2f}")
            ax2.legend()
        ax2.set_title("Detection confidence distribution")
        ax2.set_xlabel("Confidence")
        ax2.set_ylabel("Count")

        plt.suptitle(
            f"Detection overview | {focus.get('model')} / {focus.get('dataset_variant')} | "
            f"{stats.get('total_detections', 0)} total detections across {focus.get('sampled_frames', '?')} frames",
            fontsize=10, y=1.02
        )
        plt.tight_layout()
        plt.show()

        print(f"Mean confidence: {stats.get('mean_confidence', 0.0):.3f}")
        print(f"Total detections: {stats.get('total_detections', 0)}")
        print(f"Detections with track ID: {stats.get('tracked_detections', 0)}")
    else:
        print("No class statistics available in this summary.")

The chart below shows how many detections appear per sampled frame over time. Sudden spikes or drops indicate crowded scenes, camera changes, or detection instability.

In [ ]:
if focus and focus.get("mode") == "match":
    detections = focus.get("detections", [])
    timeline = focus.get("sampled_frame_timeline", [])

    if detections and timeline:
        counts_by_sample: dict[int, int] = Counter(
            int(det["sample_index"]) for det in detections if "sample_index" in det
        )
        timestamps = {item["sample_index"]: item["timestamp_seconds"] for item in timeline}

        xs = sorted(timestamps.keys())
        ts = [timestamps[i] for i in xs]
        ys = [counts_by_sample.get(i, 0) for i in xs]

        if ts:
            fig, ax = plt.subplots(figsize=(11, 3.5))
            ax.plot(ts, ys, color="#3b82f6", linewidth=1.4, alpha=0.85)
            ax.fill_between(ts, ys, alpha=0.12, color="#3b82f6")
            ax.axhline(np.mean(ys), color="#dc2626", linestyle="--", linewidth=1.2,
                       label=f"mean = {np.mean(ys):.1f}")
            ax.set_xlabel("Timestamp (s)")
            ax.set_ylabel("Detections per frame")
            ax.set_title("Detections per sampled frame over time")
            ax.legend()
            plt.tight_layout()
            plt.show()
        else:
            print("Timeline data missing timestamp information.")
    else:
        print("Detections or timeline not available.")

## 6. Tracking Quality

In [ ]:
if focus:
    tracking = focus.get("tracking", {})
    stats = focus.get("stats", {})

    track_count = stats.get("track_count", tracking.get("track_count", 0))
    tracked_detections = stats.get("tracked_detections", 0)
    total_detections = stats.get("total_detections", 0)
    sampled_frames = focus.get("sampled_frames", 0)

    print("Tracking summary")
    print(f"  Unique track IDs assigned : {track_count}")
    print(f"  Detections with track ID  : {tracked_detections} / {total_detections}")
    if sampled_frames > 0:
        print(f"  Avg detections per frame  : {total_detections / sampled_frames:.1f}")
    if track_count > 0:
        print(f"  Avg detections per track  : {total_detections / track_count:.1f}")

    primary_ball = tracking.get("primary_ball", {})
    if primary_ball:
        print(f"  Ball primary selection     : enabled={primary_ball.get('enabled')}, "
              f"memory={primary_ball.get('memory_seconds')}s")

In [ ]:
if focus and focus.get("mode") == "match":
    detections = focus.get("detections", [])
    timeline = focus.get("sampled_frame_timeline", [])
    ts_lookup = {item["sample_index"]: item["timestamp_seconds"] for item in timeline}

    track_frames: dict[int, list[float]] = defaultdict(list)
    for det in detections:
        tid = det.get("track_id")
        sidx = det.get("sample_index")
        if isinstance(tid, int) and isinstance(sidx, int):
            ts = ts_lookup.get(sidx, sidx)
            track_frames[tid].append(ts)

    if track_frames:
        lifespans = [max(v) - min(v) for v in track_frames.values()]

        fig, axes = plt.subplots(1, 2, figsize=(11, 4))

        ax = axes[0]
        ax.hist(lifespans, bins=20, color="#8b5cf6", edgecolor="white", linewidth=0.5, alpha=0.85)
        ax.axvline(np.mean(lifespans), color="#dc2626", linestyle="--", linewidth=1.5,
                   label=f"mean = {np.mean(lifespans):.1f}s")
        ax.set_title("Track lifespan distribution")
        ax.set_xlabel("Lifespan (s)")
        ax.set_ylabel("Number of tracks")
        ax.legend()

        ax2 = axes[1]
        appearance_counts = [len(v) for v in track_frames.values()]
        ax2.hist(appearance_counts, bins=20, color="#0891b2", edgecolor="white", linewidth=0.5, alpha=0.85)
        ax2.axvline(np.mean(appearance_counts), color="#dc2626", linestyle="--", linewidth=1.5,
                    label=f"mean = {np.mean(appearance_counts):.1f} frames")
        ax2.set_title("Frames per track")
        ax2.set_xlabel("Number of sampled frames")
        ax2.set_ylabel("Number of tracks")
        ax2.legend()

        plt.suptitle(f"Track ID quality | {len(track_frames)} unique tracks total", fontsize=10, y=1.02)
        plt.tight_layout()
        plt.show()

        short_lived = sum(1 for l in lifespans if l < 1.0)
        print(f"Tracks with lifespan < 1 s: {short_lived} ({100*short_lived/len(lifespans):.0f}%)")
        print(f"Tracks with lifespan > 5 s: {sum(1 for l in lifespans if l > 5.0)}")
    else:
        print("No track ID data found in detections.")

## 7. Team Assignment

In [ ]:
if focus:
    team_assignment = focus.get("team_assignment", {})
    stats = focus.get("stats", {})

    team_counts = stats.get("team_counts", team_assignment.get("team_counts", {}))
    method = team_assignment.get("method", team_assignment.get("frame_assignment_method", "?"))

    print(f"Assignment method: {method}")
    print(f"Team counts (across all detections):")
    for team, count in sorted(team_counts.items()):
        print(f"  {team}: {count}")

In [ ]:
if focus:
    detections = focus.get("detections", [])
    player_classes = {"player", "goalkeeper"}

    team_detections = Counter()
    confidence_by_team = defaultdict(list)

    for det in detections:
        cls = str(det.get("class", "")).lower()
        if cls in player_classes or cls == "referee":
            team = str(det.get("team", "unknown"))
            team_detections[team] += 1
            tc = det.get("team_confidence")
            if isinstance(tc, (int, float)):
                confidence_by_team[team].append(float(tc))

    if team_detections:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))

        ax = axes[0]
        teams = list(team_detections.keys())
        counts = [team_detections[t] for t in teams]
        colors = [TEAM_COLORS.get(t, UNKNOWN_COLOR) for t in teams]
        bars = ax.bar(teams, counts, color=colors, edgecolor="white", linewidth=0.8)
        ax.bar_label(bars, padding=3, fontsize=10)
        ax.set_title("Detections by team label")
        ax.set_ylabel("Detections")

        ax2 = axes[1]
        for team, confs in confidence_by_team.items():
            if confs:
                ax2.hist(confs, bins=15, alpha=0.6,
                         color=TEAM_COLORS.get(team, UNKNOWN_COLOR),
                         label=f"{team} (n={len(confs)})",
                         edgecolor="white", linewidth=0.4)
        ax2.set_title("Team assignment confidence")
        ax2.set_xlabel("Confidence")
        ax2.set_ylabel("Count")
        ax2.legend(fontsize=9)

        plt.suptitle(
            f"Team assignment | {focus.get('model')} / {focus.get('dataset_variant')}",
            fontsize=10, y=1.02
        )
        plt.tight_layout()
        plt.show()

        total_players = sum(v for k, v in team_detections.items() if k in {"team_a", "team_b", "referee", "unknown"})
        unknown_pct = 100 * team_detections.get("unknown", 0) / max(1, total_players)
        print(f"Unknown assignments: {team_detections.get('unknown', 0)} ({unknown_pct:.1f}% of player/goalkeeper/referee detections)")
    else:
        print("No player/goalkeeper/referee detections found.")

The stability chart below shows how consistently each track ID was assigned to a single team across all frames it appeared in. A track with 100% consistency was always assigned the same team label; a lower value indicates label flickering.

In [ ]:
if focus and focus.get("mode") == "match":
    detections = focus.get("detections", [])
    player_classes = {"player", "goalkeeper"}

    team_votes: dict[int, list[str]] = defaultdict(list)
    for det in detections:
        cls = str(det.get("class", "")).lower()
        if cls not in player_classes:
            continue
        tid = det.get("track_id")
        team = det.get("team")
        if isinstance(tid, int) and team:
            team_votes[tid].append(str(team))

    if team_votes:
        consistencies = []
        for tid, votes in team_votes.items():
            if not votes:
                continue
            most_common_count = Counter(votes).most_common(1)[0][1]
            consistencies.append(most_common_count / len(votes))

        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.hist(consistencies, bins=20, color="#f59e0b", edgecolor="white", linewidth=0.5, alpha=0.85)
        ax.axvline(np.mean(consistencies), color="#dc2626", linestyle="--", linewidth=1.5,
                   label=f"mean = {np.mean(consistencies):.2f}")
        ax.set_xlabel("Team label consistency (fraction of dominant label)")
        ax.set_ylabel("Number of tracks")
        ax.set_title("Per-track team label consistency")
        ax.legend()
        plt.tight_layout()
        plt.show()

        stable = sum(1 for c in consistencies if c >= 0.9)
        print(f"Tracks with consistency >= 0.9: {stable}/{len(consistencies)} ({100*stable/len(consistencies):.0f}%)")
    else:
        print("No player tracks found for consistency analysis.")

## 8. Minimap Entities

The minimap in Murawa is a list of entities per sampled frame, stored in `prediction_summary.json` under `minimap_entities`. Each entity records the class, team, and a normalized field position inferred from the bounding box center. This data drives the Minimap tab in the Streamlit app; it is not drawn as a graphical layer on the output video.

On some clips the field can be empty if no pitch homography is computed or if the pipeline produces no valid player positions for that frame.

In [ ]:
if focus:
    minimap_entities = focus.get("minimap_entities", [])
    detections = focus.get("detections", [])

    print(f"Minimap entities in summary : {len(minimap_entities)}")
    print(f"Total detections in summary : {len(detections)}")

    if minimap_entities:
        entity_classes = Counter(str(e.get("class", "?")) for e in minimap_entities)
        entity_teams = Counter(str(e.get("team", "?")) for e in minimap_entities)
        print("Entity class breakdown:", dict(entity_classes))
        print("Entity team breakdown: ", dict(entity_teams))

        has_pos = sum(
            1 for e in minimap_entities
            if e.get("x") is not None or e.get("position") is not None
        )
        print(f"Entities with position data: {has_pos} / {len(minimap_entities)}")
    else:
        print("No minimap entities in this summary.")
        print("This is expected when prediction_summary.json comes from a frame that the pipeline")
        print("could not map to pitch coordinates, or when the clip contained no detectable players.")

In [ ]:
if focus and focus.get("minimap_entities"):
    entities = focus["minimap_entities"]

    def extract_xy(entity: dict):
        if "x" in entity and "y" in entity:
            return entity["x"], entity["y"]
        pos = entity.get("position")
        if isinstance(pos, (list, tuple)) and len(pos) == 2:
            return pos[0], pos[1]
        return None, None

    plotable = [(e, *extract_xy(e)) for e in entities]
    plotable = [(e, x, y) for e, x, y in plotable if x is not None and y is not None]

    if plotable:
        fig, ax = plt.subplots(figsize=(9, 5))
        field = plt.Rectangle((0, 0), 1, 1, fill=True, facecolor="#16a34a", edgecolor="white", linewidth=2)
        ax.add_patch(field)
        ax.axvline(0.5, color="white", linewidth=1.2, alpha=0.7)
        centre = plt.Circle((0.5, 0.5), 0.1, fill=False, edgecolor="white", linewidth=1.0, alpha=0.6)
        ax.add_patch(centre)

        for ent, x, y in plotable:
            team = str(ent.get("team", "unknown"))
            cls = str(ent.get("class", "")).lower()
            color = TEAM_COLORS.get(team, UNKNOWN_COLOR)
            marker = "o" if cls not in ("ball",) else "*"
            size = 40 if cls != "ball" else 120
            ax.scatter(x, y, c=color, s=size, marker=marker, edgecolors="white",
                       linewidths=0.5, alpha=0.85, zorder=3)

        legend_items = [
            mpatches.Patch(color=TEAM_A_COLOR, label="team_a"),
            mpatches.Patch(color=TEAM_B_COLOR, label="team_b"),
            mpatches.Patch(color=REFEREE_COLOR, label="referee"),
            mpatches.Patch(color=UNKNOWN_COLOR, label="unknown"),
        ]
        ax.legend(handles=legend_items, loc="upper right", fontsize=9, framealpha=0.8)
        ax.set_xlim(-0.05, 1.05)
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(f"Minimap entity positions ({len(plotable)} plotted)")
        ax.set_xlabel("Normalized field X")
        ax.set_ylabel("Normalized field Y")
        ax.set_aspect("equal")
        plt.tight_layout()
        plt.show()
    else:
        print("Minimap entities lack coordinate data in this summary.")

## 9. Successes and Failures

This section summarizes what the pipeline handles reliably and where it consistently falls short, based on inspection of saved summaries and rendered videos.

In [ ]:
if summaries:
    all_match = [s for s in summaries if s.get("mode") == "match"]

    if all_match:
        metrics = []
        for s in all_match:
            stats = s.get("stats", {})
            det = s.get("detections", [])

            player_classes = {"player", "goalkeeper"}
            team_votes = defaultdict(list)
            for d in det:
                cls = str(d.get("class", "")).lower()
                if cls in player_classes:
                    tid = d.get("track_id")
                    team = d.get("team")
                    if isinstance(tid, int) and team:
                        team_votes[tid].append(str(team))

            consistencies = []
            for votes in team_votes.values():
                if votes:
                    most_common = Counter(votes).most_common(1)[0][1]
                    consistencies.append(most_common / len(votes))

            team_counts = stats.get("team_counts", {})
            total_players = sum(v for k, v in team_counts.items() if k in {"team_a", "team_b", "unknown"})
            unknown_pct = 100 * team_counts.get("unknown", 0) / max(1, total_players)

            metrics.append({
                "run": s.get("resolved_run_name", "")[-25:],
                "model": s.get("model", "?"),
                "variant": s.get("dataset_variant", "?"),
                "mean_conf": round(stats.get("mean_confidence", 0.0), 3),
                "track_count": stats.get("track_count", 0),
                "mean_team_consistency": round(np.mean(consistencies), 3) if consistencies else None,
                "unknown_team_pct": round(unknown_pct, 1),
                "minimap_entities": len(s.get("minimap_entities", [])),
            })

        df_metrics = pd.DataFrame(metrics)
        display(df_metrics)
    else:
        print("No match-mode summaries found for the comparative table.")

### What works well

**Player detection** is the strongest part of the system. Both YOLO and RF-DETR detect players reliably at typical broadcast distances, and confidence scores are generally high. The detection rate is stable across most parts of a clip.

**ByteTrack tracking** assigns consistent IDs across frames for players who remain visible and do not overlap significantly with other players. Long-lived tracks (appearing across many sampled frames) are common for central field players.

**Clip-level team stabilization** is a meaningful improvement over per-frame clustering. Without it, the `team_a`/`team_b` labels can swap between frames whenever two groups of players happen to cluster differently. The prototype-based approach anchors labels to the two dominant jersey colors computed over the full clip.

**Pipeline robustness** is solid: the system handles missing inputs gracefully, falls back cleanly on errors, and writes structured JSON outputs that are easy to inspect programmatically.

### Where the system struggles

**Ball detection** is the most unreliable class. The ball is small and fast, and the model misses it frequently in motion-blurred or low-contrast frames. The 2-second memory mechanism in `tracking_ball.py` partially compensates for short gaps, but on clips with significant camera movement the ball track is often fragmented.

**Team assignment on crowded or zoomed-out frames** degrades because bounding boxes become too small to extract a reliable jersey crop. The minimum box size thresholds (`MIN_PLAYER_BOX_WIDTH_PX = 15`, `MIN_PLAYER_BOX_HEIGHT_PX = 35`) are intentionally conservative, which means distant players are frequently assigned `unknown` rather than a potentially wrong team label.

**Tracking fragmentation during occlusion** causes high numbers of short-lived track IDs. A player who briefly goes behind another player or the camera cuts to a different angle gets a new track ID. This inflates `track_count` beyond the number of actual players on the pitch.

**Goalkeepers** are a separate class but are treated like players for team assignment, which can distort clustering if the goalkeeper has an unusual jersey color compared to outfield players.

**The minimap** is a JSON entity list, not a drawn pitch view on the video. Positions rely on the bounding box center projected to a normalized field coordinate, without actual homography from image to pitch. The result is an approximate positional signal only and the Streamlit tab can show an empty pitch if no suitable detections are present.

## 10. Limitations of the Demonstration Version

The current implementation is a demonstration, not a production system. The following constraints apply:

**Video length.** Input videos are capped at 60 seconds (`max_video_duration_seconds` in `configs/project.yaml`). Longer clips are rejected at validation time.

**Sampling rate.** The default sampling rate is 3 FPS. At this rate a 60-second clip yields 180 frames. The temporal resolution is sufficient for tracking over short distances but not for fast events like shots or quick passes. The rate can be increased up to 24 FPS from the Streamlit UI, but this increases processing time linearly.

**No real-time processing.** The pipeline is batch-mode only. There is no live stream input and no incremental output.

**Team assignment is unsupervised.** There are no jersey number labels, team roster data, or calibration step. The system relies entirely on jersey color clustering, which fails when two teams wear similar colors or when only one team is visible in the clip.

**No pitch homography.** The minimap positions are not calibrated to actual pitch coordinates. The system does not identify the pitch markings or compute a perspective transform, so positional data is only approximate.

**Single camera.** The pipeline processes a single video feed. There is no multi-camera fusion and no handling of broadcast cuts or replays, which cause abrupt tracking resets.

**No re-identification.** Once a player loses their track ID (due to occlusion, a camera cut, or leaving the frame), they receive a new ID when they reappear. This means the total number of unique track IDs over a 60-second clip can easily exceed the number of players on the pitch.

**Referee handling.** Referees are detected and labeled correctly but are not excluded from the player count in team statistics in all code paths.

## 11. Output Artifacts

In [ ]:
pred_dirs = sorted(PREDICTIONS_DIR.rglob("prediction_summary.json"))
video_files = sorted(VIDEOS_DIR.glob("*.mp4")) + sorted(VIDEOS_DIR.glob("*.webm"))

print(f"Prediction summaries: {len(pred_dirs)}")
for p in pred_dirs:
    size_kb = p.stat().st_size // 1024
    print(f"  {p.relative_to(PROJECT_ROOT)}  ({size_kb} KB)")

print(f"Output videos: {len(video_files)}")
for v in video_files:
    size_mb = v.stat().st_size / (1024 * 1024)
    print(f"  {v.relative_to(PROJECT_ROOT)}  ({size_mb:.1f} MB)")

In [ ]:
if focus:
    top_level_keys = list(focus.keys())
    top_level_keys = [k for k in top_level_keys if not k.startswith("_")]

    print("Top-level keys in prediction_summary.json:")
    for key in top_level_keys:
        val = focus[key]
        if isinstance(val, list):
            print(f"  {key}: list of {len(val)} items")
        elif isinstance(val, dict):
            print(f"  {key}: dict with keys {list(val.keys())[:6]}")
        else:
            print(f"  {key}: {repr(val)[:60]}")

## 12. Conclusions

The Murawa pipeline delivers a working end-to-end demonstration of football video analysis. Starting from a short video clip, it reliably detects players and referees, maintains consistent track identities across most of the clip, and correctly separates the two teams by jersey color in the majority of cases.

The most robust component is player detection. Both the YOLO and RF-DETR models trained on the SoccerNet dataset perform well at typical broadcast resolutions, and the pipeline infrastructure around them (frame extraction, structured JSON output, Streamlit integration) works cleanly.

The weakest points in the current version are ball tracking and the minimap. Ball detection is inherently harder due to the object's small size and motion blur, and the minimap remains a list of approximate positions rather than a calibrated pitch view. These are known design limitations of the demo scope rather than implementation bugs.

Team assignment quality varies with clip content. It works well when both teams are visible in similar proportions and wear clearly distinct jerseys. It degrades on zoomed-out shots where bounding boxes become too small for reliable jersey color extraction, and it does not handle goalkeeper jerseys (which often differ from outfield players) as a special case.

As a demonstration project completed in a single semester, the system achieves its stated goals: stable object detection, basic tracking, team separation, and a Streamlit UI that integrates all components into a usable interface. The architecture is structured well for future extension: each component is isolated, the output format is consistent, and the configuration system makes it straightforward to adjust inference parameters or swap models.